The project involves training and evaluating transformer-based classifiers for a text classification task. You should compare the following three approaches:

1. Prompting a generative (GPT-like) model
2. Fine-tuning a generative (GPT-like) model
3. Fine-tuning a bidirectional (BERT-like) model

Specifically, you should complete the following steps

1. Select a text classification corpus to work on. (see below)
2. Read the paper presenting the corpus as well as any other relevant published materials about the corpus to assure you understand the data.
3. Identify what is the random baseline performance (selecting the label randomly) as well as expected performance for recent machine learning models for this corpus. The paper describing the data may help you here.
4. Select a generative model and a bidirectional model to work on. (see below)
5. Write code to
- download the corpus
- perform any necessary sampling and preprocessing (see below)
- for each approach (1, 2, and 3), optimize your method while evaluating performance on the validation set
  - prompting approach (1): optimize your prompt, taking any few-shot examples from the training set
  - fine-tuning approaches (2 and 3): perform hyperparameter optimization while training on the training set
- evaluate the approaches on the test set
6. Write a summary of
- what you learned about the corpus
- your results
- how your results relate to the state of the art (i.e. the best published results) / expected performance
- if completed as a group project, a section detailing who did what

# 1. Preparations and baseline

## 1.1. Imports, datasets and models

In [1]:
!pip3 install -q datasets transformers evaluate accelerate
import datasets, torch, transformers, evaluate
import matplotlib.pyplot as plt
import random
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
dataset = datasets.load_dataset('imdb')
del dataset["unsupervised"]

#downsample and stratify test data
dataset["test"]=dataset["test"].train_test_split(
    test_size=400,
    stratify_by_column="label",
    shuffle=True,
    seed=42
)["test"]

#pick 2 few-shot examples from train set
split = dataset["train"].train_test_split(
    test_size=2,
    stratify_by_column="label",
    shuffle=True,
    seed=42
)
dataset["train"] = split["train"]
fewshot_examples = split["test"]

#check label distributions for soon-to-be-used datasets
print(Counter(dataset["test"]["label"]))
print(Counter(fewshot_examples["label"]))
print(Counter(dataset["train"]["label"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Counter({1: 200, 0: 200})
Counter({0: 1, 1: 1})
Counter({0: 12499, 1: 12499})


In [3]:
#a few smaller datasets are used for the gpt-like model due to platform resource constraints
#since training times are long (especially without GPU) we limit the training set size
train_small = dataset["train"].train_test_split(
    train_size=1000,
    stratify_by_column="label",
    shuffle=True,
    seed=42
)["train"]

#we use a smaller eval set as well due to colab memory maxing out otherwise
eval_small = dataset["test"].train_test_split(
    test_size=70,
    stratify_by_column="label",
    shuffle=True,
    seed=42
)["test"]

In [4]:
#import models
bert_model_name = "bert-base-cased"
gen_model_name = "HuggingFaceTB/SmolLM-135M-Instruct"
gen_model = transformers.AutoModelForCausalLM.from_pretrained(gen_model_name)
bert_model = transformers.AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=2)

#define gpt-like tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(gen_model_name, use_fast=True)

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

## 1.2. Random baseline performance

In [7]:
#use the training set as it's the largest that we defined earlier
y_true = []
y_pred = []

for sample in dataset["train"]:
  pred = random.randint(0, 1)
  y_true.append(sample["label"])
  y_pred.append(pred)

base_acc = accuracy_score(y_true, y_pred)
print("random baseline: ", base_acc)

random baseline:  0.501960156812545


# 2. GPT-like approach

## 2.1. Prompting (zero-/few-shot)

In [5]:
#method for handling prompts
def format_prompt(prompt, example):
    prompt += f"""You are a sentiment classifier.
        Classify the following review as either positive (1) or negative (0)
        Review: {example["text"]}
        Sentiment:"""
    return prompt

In [ ]:
#zero-shot approach

y_true = []
y_pred = []

prompt = ""

for sample in dataset["test"]:
  inputs = tokenizer(format_prompt(prompt, sample), return_tensors="pt").to(gen_model.device)
  with torch.no_grad():
    outputs = gen_model(**inputs)

  logits = outputs.logits[0, -1]
  id_0 = tokenizer.encode("0", add_special_tokens=False)[0]
  id_1 = tokenizer.encode("1", add_special_tokens=False)[0]
  pred = 1 if logits[id_1] > logits[id_0] else 0

  y_true.append(sample["label"])
  y_pred.append(pred)

accuracy_zeroshot = accuracy_score(y_true, y_pred)
print("zero-shot accuracy: ", accuracy_zeroshot)

zero-shot accuracy:  0.53


In [ ]:
# few-shot approach

y_true = []
y_pred = []

prompt = f"""
        Review: {fewshot_examples["text"][0]}.
        Sentiment: {fewshot_examples["label"][0]}.

        Review: {fewshot_examples["text"][1]}.
        Sentiment: {fewshot_examples["label"][1]} \n"""

for sample in dataset["test"]:
  inputs = tokenizer(format_prompt(prompt, sample), return_tensors="pt").to(gen_model.device)
  with torch.no_grad():
    outputs = gen_model(**inputs)

  logits = outputs.logits[0, -1]
  id_0 = tokenizer.encode("0", add_special_tokens=False)[0]
  id_1 = tokenizer.encode("1", add_special_tokens=False)[0]
  pred = 1 if logits[id_1] > logits[id_0] else 0

  y_true.append(sample["label"])
  y_pred.append(pred)

accuracy_fewshot = accuracy_score(y_true, y_pred)
print("few-shot accuracy: ", accuracy_fewshot)

few-shot accuracy:  0.56


## 2.2. Fine-tuning

In [6]:
#define functions for dataset tokenization
def tokenize(example):
    text = format_prompt("", example)
    tokenized = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    #create the labels tensor
    labels = [-100] * len(tokenized["input_ids"])
    sentiment_position = sum(tokenized["attention_mask"]) - 1
    if example["label"] == 0:
        labels[sentiment_position] = tokenizer.encode("0", add_special_tokens=False)[0]
    else:
        labels[sentiment_position] = tokenizer.encode("1", add_special_tokens=False)[0]

    tokenized["labels"] = labels
    return tokenized

In [10]:
#tokenize the smaller datasets for the gpt-like model
train_small = train_small.map(tokenize, remove_columns=train_small.column_names)
eval_small = eval_small.map(tokenize, remove_columns=eval_small.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

In [11]:
#function for accuracy calculations
accuracy = evaluate.load("accuracy")
def compute_gpt_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = outputs.argmax(axis=-1)

    y_true = []
    y_pred = []
    id_0 = tokenizer.encode("0", add_special_tokens=False)[0]
    id_1 = tokenizer.encode("1", add_special_tokens=False)[0]

    for i in range(len(labels)):
        label_idxs = (labels[i] != -100).nonzero()

        if label_idxs[0].size > 0:
            tgt = label_idxs[0][0].item()
            id_true = labels[i, tgt].item()
            id_pred = predictions[i, tgt].item()

            if id_true == id_0:
                y_true.append(0)
            else:
                y_true.append(1)

            if id_pred == id_0:
                y_pred.append(0)
            else:
                y_pred.append(1)

    return accuracy.compute(predictions=y_pred, references=y_true)

In [13]:
torch.cuda.empty_cache()

training_args = transformers.TrainingArguments(
    output_dir="./gen_finetuned",
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    load_best_model_at_end=True,
    eval_steps=100,
    logging_steps=100,
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    max_steps=1000,
    fp16=True,
    report_to="none",
)

trainer = transformers.Trainer(
    model=gen_model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=eval_small,
    compute_metrics=compute_gpt_accuracy,
    data_collator=transformers.DataCollatorWithPadding(tokenizer),
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model("./gen_finetuned")

/tmp/ipython-input-2599563737.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = transformers.Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy
100,1.242500,0.953812,0.585714
200,1.340200,1.184652,0.557143
300,1.679600,0.947025,0.571429
400,1.106800,2.337603,0.557143
500,0.843000,0.741009,0.742857
600,1.126600,1.080166,0.757143
700,0.610200,1.158959,0.800000
800,0.923500,0.811237,0.842857
900,0.729600,0.965725,0.828571
1000,0.730500,0.815224,0.828571


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

In [14]:
gpt_results = trainer.evaluate(eval_small)
print(gpt_results)
print('Accuracy:', gpt_results['eval_accuracy'])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7410092949867249, 'eval_accuracy': 0.7428571428571429, 'eval_runtime': 251.9675, 'eval_samples_per_second': 0.278, 'eval_steps_per_second': 0.278, 'epoch': 1.0}
Accuracy: 0.7428571428571429


# 3. BERT-like approach

In [6]:
#update the tokenizer and tokenization from gpt-like to bert-like
tokenizer = transformers.AutoTokenizer.from_pretrained(bert_model_name, use_fast=True)
def tokenize(example):
    text = format_prompt("", example)
    return tokenizer(
        text,
        max_length=512,
        truncation=True,
    )
dataset_tokenized = dataset.map(tokenize)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/24998 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [7]:
#function for bert-like model accuracy calculations
accuracy = evaluate.load("accuracy")
def compute_bert_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = outputs.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [8]:
torch.cuda.empty_cache()

trainer_args = transformers.TrainingArguments(
    "checkpoints",
    eval_strategy="steps",
    logging_strategy="steps",
    load_best_model_at_end=True,
    eval_steps=100,
    logging_steps=100,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    max_steps=1000,
    report_to="none",
)

trainer = transformers.Trainer(
    model=bert_model,
    args=trainer_args,
    train_dataset=dataset_tokenized["train"],
    eval_dataset=dataset_tokenized["test"],
    compute_metrics=compute_bert_accuracy,
    data_collator=transformers.DataCollatorWithPadding(tokenizer),
    tokenizer=tokenizer
)

trainer.train()
trainer.save_model("./bert_finetuned")

/tmp/ipython-input-544914054.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = transformers.Trainer(


Step,Training Loss,Validation Loss,Accuracy
100,0.588200,0.580168,0.712500
200,0.433800,0.303776,0.885000
300,0.416400,0.334193,0.872500
400,0.320300,0.340262,0.900000
500,0.349400,0.331539,0.902500
600,0.316700,0.356033,0.887500
700,0.320400,0.275853,0.902500
800,0.253200,0.289633,0.907500
900,0.282200,0.274935,0.915000
1000,0.325900,0.267265,0.920000


In [9]:
bert_results = trainer.evaluate(dataset_tokenized["test"])
print(bert_results)
print('Accuracy:', bert_results['eval_accuracy'])

{'eval_loss': 0.267264723777771, 'eval_accuracy': 0.92, 'eval_runtime': 12.0027, 'eval_samples_per_second': 33.326, 'eval_steps_per_second': 1.083, 'epoch': 0.32}
Accuracy: 0.92


# 4. Summary

The project was made using the IMDB movie review dataset, on which the random baseline accuracy was identified to be ~50%.

In the dataset, a lot of the reviews included summarization or explanations of the movies or serieses themselves which doesn't really give the models much to work with sentiment-wise, but as the reviews are either notably positive or negative (no reviews that give just an average rating) there are enough subjective cues so that the model should be able to make a guided decision instead of a random guess.

The results that I got were most likely quite uncertain, especially for the gpt-like model as I had to heavily limit the size of the training and test/eval sets due to colab resource constraints. However, the results are as follows:
- Prompting:
  - Zero-shot: 53,0%
  - Few-shot: 56,0%
- Fine-tuned GPT-like (SmolLM-135M-Instruct): 74,3%
- Fine-tuned BERT-like (bert-base-cased): 92,0%

When compared to state-of-the-art (sota), even after hyperparameter tuning these models don't can't really match them due to either poor quality prompts (max_length in fine-tuning either 256 or 512, which may lead to reviews being cut short), or the lack of usable at-hand data due to, again, colab constraints. The best models for the task of classification on this dataset seem to have an accuracy of around 96-97%, and although most of the best results use a bert-like model, some also are gpt-based or a combination of just one or both of them, as seen in for example the number 1 model.

I personally expected the results of the fine-tuned models to be at least 80-90%, so both of the fine-tuned models surprised me, one positively and one negatively. For the zero- or few-shot tests I didn't expect much, but to be just above random guesses in accuracy is a bit surprising to me.

Link to referred-to sota-results: https://opencodepapers-b7572d.gitlab.io/benchmarks/sentiment-analysis-on-imdb.html

